In [131]:
from pyspark.sql import SparkSession
spark=SparkSession.builder \
     .appName("Working with files") \
     .getOrCreate()

with csv

In [132]:
%%writefile bookings.csv
booking_id,customer_name,city,service_type,provider,booking_amount,booking_status,payment_mode
1001,Aarav Mehta,Hyderabad,Flight,IndiGo,6500,Confirmed,UPI
1002,Sana Khan,Bangalore,Hotel,Pearl Grand,4500,Confirmed,Card
1003,John Mathew,,Flight,Air India,12000,Confirmed,UPI
1004,Ayesha Begum,Hyderabad,Hotel,,7500,Pending,Cash
1005,Vikram Rao,Mumbai,Flight,Vistara,,Confirmed,Card
1006,Divya Sharma,Delhi,Flight,IndiGo,5900,Cancelled,
1007,Imran Ali,Pune,Hotel,Budget Inn,2200,,UPI
1008,Meera Nair,Kochi,Hotel,Hill View Resort,7500,Confirmed,Card
1009,Rohan Das,Kolkata,Flight,Air India,7400,Pending,UPI
1010,Nisha Reddy,Bangalore,Flight,British Airways,62000,Confirmed,Card
1011,Farhan Ali,,Hotel,Skyline Suites,22000,Confirmed,
1012,Neha Singh,Hyderabad,,Emirates,28000,Confirmed,UPI
1013,Arjun Verma,Chennai,Flight,,15000,Cancelled,Cash
1014,Kavya Nair,Mumbai,Hotel,Sea View Stay,,Pending,Card
1015,Ravi Kumar,Delhi,Flight,SpiceJet,4800,Confirmed,UPI

Overwriting bookings.csv


In [133]:
df = spark.read.csv("bookings.csv", header=True, inferSchema=True)

In [134]:
df.printSchema()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- provider: string (nullable = true)
 |-- booking_amount: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_mode: string (nullable = true)



In [135]:
df.count()

15

In [136]:
df.filter(df["city"].isNull()).show()

+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|      1003|  John Mathew|NULL|      Flight|     Air India|         12000|     Confirmed|         UPI|
|      1011|   Farhan Ali|NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+



In [137]:
df.filter(df["provider"].isNull()).show()

+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|      1004| Ayesha Begum|Hyderabad|       Hotel|    NULL|          7500|       Pending|        Cash|
|      1013|  Arjun Verma|  Chennai|      Flight|    NULL|         15000|     Cancelled|        Cash|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+



In [138]:
df.filter(df["booking_amount"].isNull()).show()

+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|booking_id|customer_name|  city|service_type|     provider|booking_amount|booking_status|payment_mode|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|      1005|   Vikram Rao|Mumbai|      Flight|      Vistara|          NULL|     Confirmed|        Card|
|      1014|   Kavya Nair|Mumbai|       Hotel|Sea View Stay|          NULL|       Pending|        Card|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+



In [139]:
df.filter(df["booking_status"].isNull()).show()

+----------+-------------+----+------------+----------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|  provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+----------+--------------+--------------+------------+
|      1007|    Imran Ali|Pune|       Hotel|Budget Inn|          2200|          NULL|         UPI|
+----------+-------------+----+------------+----------+--------------+--------------+------------+



In [140]:
df.filter(df["payment_mode"].isNull()).show()

+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name| city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|      1006| Divya Sharma|Delhi|      Flight|        IndiGo|          5900|     Cancelled|        NULL|
|      1011|   Farhan Ali| NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+



In [141]:
from pyspark.sql.functions import col, count, when
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+----------+-------------+----+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------+--------------+--------------+------------+
|         0|            0|   2|           1|       2|             2|             1|           2|
+----------+-------------+----+------------+--------+--------------+--------------+------------+



In [142]:
df_no_nulls = df.dropna()

In [143]:
df_clean_amount = df.dropna(subset=["booking_amount"]).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL|         UPI|
|      100

In [144]:
df_clean_critical = df.dropna(subset=["customer_name", "service_type", "booking_amount"]).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL|         UPI|
|      100

In [145]:
df = df.na.fill({"city": "Unknown"})

In [146]:
df = df.fillna({"provider": "Not Available"})
df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [147]:
df = df.fillna({"payment_mode": "Not Provided"})

In [148]:
df = df.fillna({"booking_status": "Unknown"})

In [149]:
df = df.fillna({"booking_amount": 0})

In [150]:
from pyspark.sql.functions import when, col
null_cond = col(df.columns[0]).isNull()
for c in df.columns[1:]:
    null_cond = null_cond | col(c).isNull()
df_quality = df.withColumn("data_quality_status", when(null_cond, "Incomplete").otherwise("Complete"))
df_quality.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|           Complete|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Conf

In [151]:
df_quality.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   14|
|         Incomplete|    1|
+-------------------+-----+



In [152]:
df_quality.filter(col("data_quality_status") == "Complete").show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|           Complete|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Conf

In [153]:
df_quality.filter(col("data_quality_status") == "Incomplete").show()

+----------+-------------+---------+------------+--------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+-------------------+
|      1012|   Neha Singh|Hyderabad|        NULL|Emirates|         28000|     Confirmed|         UPI|         Incomplete|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+-------------------+



In [154]:
trans_df = df.withColumn("tax", col("booking_amount") * 0.05).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|   tax|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI| 325.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card| 225.0|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI| 600.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash| 375.0|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Confirmed|        Card|   0.0|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiG

In [160]:
from pyspark.sql.functions import col

df = df.withColumn(
    "tax",
    col("booking_amount") * 0.05
).withColumn(
    "final_amount",
    col("booking_amount") + col("tax")
)

In [161]:
from pyspark.sql.functions import sum
df.filter(col("booking_status") == "Confirmed").select(sum("booking_amount")).show()

+-------------------+
|sum(booking_amount)|
+-------------------+
|             147300|
+-------------------+



In [162]:
df.groupBy("service_type").count().show()

+------------+-----+
|service_type|count|
+------------+-----+
|        NULL|    1|
|       Hotel|    6|
|      Flight|    8|
+------------+-----+



In [163]:
df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|  Kolkata|    1|
|  Unknown|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [164]:
from pyspark.sql.functions import avg
df_filled_zero = df.fillna({"booking_amount": 0})
df_filled_zero.select(avg("booking_amount")).show()

+-------------------+
|avg(booking_amount)|
+-------------------+
| 12353.333333333334|
+-------------------+



In [165]:
df_dropped_null = df.dropna(subset=["booking_amount"])
df_dropped_null.select(avg("booking_amount")).show()

+-------------------+
|avg(booking_amount)|
+-------------------+
| 12353.333333333334|
+-------------------+



In [166]:
print("Average (with 0s):")
df_filled_zero.select(avg("booking_amount")).show()

print("Average (dropped nulls):")
df_dropped_null.select(avg("booking_amount")).show()

Average (with 0s):
+-------------------+
|avg(booking_amount)|
+-------------------+
| 12353.333333333334|
+-------------------+

Average (dropped nulls):
+-------------------+
|avg(booking_amount)|
+-------------------+
| 12353.333333333334|
+-------------------+



In [167]:
df.write.mode("overwrite").parquet("clean_bookings.parquet")

with json

In [176]:
%%writefile customers.json
[
{
"customer_id": 1,
"name": "Aarav Mehta",
"city": "Hyderabad",
"membership": "Gold",
"contact": {
"phone": "9876500011",
"email": "aarav@mail.com"
},
"preferences": {
"preferred_service": "Flight",
"budget_range": "Medium"
}
},
{
"customer_id": 2,
"name": "Sana Khan",
"city": "Bangalore",
"membership": "Silver",
"contact": {
"phone": null,
"email": "sana@mail.com"
},
"preferences": {
"preferred_service": "Hotel",
"budget_range": null
}
},
{
"customer_id": 3,
"name": "John Mathew",
"city": null,
"membership": "Gold",
"contact": {
"phone": "9876500013",
"email": null
},
"preferences": {
"preferred_service": "Flight",
"budget_range": "High"
}
},
{
"customer_id": 4,
"name": "Ayesha Begum",
"city": "Hyderabad",
"membership": null,
"contact": {
"phone": "9876500014",
"email": "ayesha@mail.com"
},
"preferences": {
"preferred_service": null,
"budget_range": "Low"
}
},
{
"customer_id": 5,
"name": "Vikram Rao",
"city": "Mumbai",
"membership": "Platinum",
"contact": {
"phone": null,
"email": null
},
"preferences": {
"preferred_service": "Flight",
"budget_range": "High"
}
}
]

Overwriting customers.json


In [177]:
customers_df = spark.read.option(
    "multiline",
    "true"
).json("customers.json")

customers_df.printSchema()
customers_df.show(truncate=False)

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |


In [178]:
flat_customers_df = customers_df.select(
"customer_id",
"name",
"city",
"membership",
col("contact.phone").alias("phone"),
col("contact.email").alias("email"),
col("preferences.preferred_service").alias("preferred_service"),
col("preferences.budget_range").alias("budget_range")
)
flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [179]:
flat_customers_df = customers_df.select(
"customer_id",
"name",
"city",
"membership",
col("contact.phone").alias("phone"),
col("contact.email").alias("email"),
col("preferences.preferred_service").alias("preferred_service"),
col("preferences.budget_range").alias("budget_range")
)
flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [183]:
df_cust = spark.read.json("customers.json")

In [185]:
df_cust = spark.read.option("multiline", "true").json("customers.json")

df_cust.printSchema()
df_cust.show(truncate=False)

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |


In [188]:
from pyspark.sql.functions import col

df_flat = df_cust.select(
    col("customer_id"),
    col("name").alias("customer_name"),
    col("city"),
    col("membership"),
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    col("preferences.preferred_service").alias("preferred_service"),
    col("preferences.budget_range").alias("budget_range")
)

df_flat.show(truncate=False)

+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|customer_name|city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+
|1          |Aarav Mehta  |Hyderabad|Gold      |9876500011|aarav@mail.com |Flight           |Medium      |
|2          |Sana Khan    |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |
|3          |John Mathew  |NULL     |Gold      |9876500013|NULL           |Flight           |High        |
|4          |Ayesha Begum |Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
|5          |Vikram Rao   |Mumbai   |Platinum  |NULL      |NULL           |Flight           |High        |
+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+



In [189]:
df_flat.select("customer_name", "city", "phone", "email").show()

+-------------+---------+----------+---------------+
|customer_name|     city|     phone|          email|
+-------------+---------+----------+---------------+
|  Aarav Mehta|Hyderabad|9876500011| aarav@mail.com|
|    Sana Khan|Bangalore|      NULL|  sana@mail.com|
|  John Mathew|     NULL|9876500013|           NULL|
| Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|   Vikram Rao|   Mumbai|      NULL|           NULL|
+-------------+---------+----------+---------------+



In [190]:
df_flat.filter(col("city").isNull()).show()

+-----------+-------------+----+----------+----------+-----+-----------------+------------+
|customer_id|customer_name|city|membership|     phone|email|preferred_service|budget_range|
+-----------+-------------+----+----------+----------+-----+-----------------+------------+
|          3|  John Mathew|NULL|      Gold|9876500013| NULL|           Flight|        High|
+-----------+-------------+----+----------+----------+-----+-----------------+------------+



In [191]:
df_flat.filter(col("phone").isNull()).show()

+-----------+-------------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|customer_name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+-------------+---------+----------+-----+-------------+-----------------+------------+
|          2|    Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
|          5|   Vikram Rao|   Mumbai|  Platinum| NULL|         NULL|           Flight|        High|
+-----------+-------------+---------+----------+-----+-------------+-----------------+------------+



In [192]:
df_flat.filter(col("email").isNull()).show()

+-----------+-------------+------+----------+----------+-----+-----------------+------------+
|customer_id|customer_name|  city|membership|     phone|email|preferred_service|budget_range|
+-----------+-------------+------+----------+----------+-----+-----------------+------------+
|          3|  John Mathew|  NULL|      Gold|9876500013| NULL|           Flight|        High|
|          5|   Vikram Rao|Mumbai|  Platinum|      NULL| NULL|           Flight|        High|
+-----------+-------------+------+----------+----------+-----+-----------------+------------+



In [193]:
df_flat.filter(col("membership").isNull()).show()

+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|customer_name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+
|          4| Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+



In [194]:
df_flat.filter(col("preferred_service").isNull()).show()

+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|customer_name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+
|          4| Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+



In [195]:
df_flat.filter(col("budget_range").isNull()).show()

+-----------+-------------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|customer_name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+-------------+---------+----------+-----+-------------+-----------------+------------+
|          2|    Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
+-----------+-------------+---------+----------+-----+-------------+-----------------+------------+



In [196]:
df_flat.select([count(when(col(c).isNull(), c)).alias(c) for c in df_flat.columns]).show()

+-----------+-------------+----+----------+-----+-----+-----------------+------------+
|customer_id|customer_name|city|membership|phone|email|preferred_service|budget_range|
+-----------+-------------+----+----------+-----+-----+-----------------+------------+
|          0|            0|   1|         1|    2|    2|                1|           1|
+-----------+-------------+----+----------+-----+-----+-----------------+------------+



In [197]:
df_clean = df_flat.fillna({
    "city": "Unknown",
    "membership": "Standard",
    "phone": "Not Provided",
    "email": "Not Provided",
    "preferred_service": "Not Selected",
    "budget_range": "Unknown"
})

In [198]:
quality_cond = (
    col("city").isNull() |
    col("phone").isNull() |
    col("email").isNull() |
    col("membership").isNull() |
    col("preferred_service").isNull()
)
df_clean = df_clean.withColumn("customer_quality_status", when(quality_cond, "Inc").otherwise("Complete"))

In [199]:
df_clean.groupBy("customer_quality_status").count().show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    5|
+-----------------------+-----+



In [200]:
df_clean.filter(col("customer_quality_status") == "Complete").show()

+-----------+-------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|customer_id|customer_name|     city|membership|       phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+-------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|          1|  Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|               Complete|
|          2|    Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|               Complete|
|          3|  John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|               Complete|
|          4| Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|               Complete|
|          5|   Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not P

In [201]:
df_clean.filter(col("customer_quality_status") == "Inc").show()

+-----------+-------------+----+----------+-----+-----+-----------------+------------+-----------------------+
|customer_id|customer_name|city|membership|phone|email|preferred_service|budget_range|customer_quality_status|
+-----------+-------------+----+----------+-----+-----+-----------------+------------+-----------------------+
+-----------+-------------+----+----------+-----+-----+-----------------+------------+-----------------------+



In [202]:
df_clean.groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [203]:
df_clean.groupBy("preferred_service").count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [204]:
df_flat.write.mode("overwrite").parquet("customers_flat.parquet")

In [205]:
df_clean.write.mode("overwrite").option("header", "true").csv("clean_customers.csv")

In [206]:
original_count = df_cust.count()
clean_count = df_clean.count()
print(f"Original Count: {original_count} | Clean Count: {clean_count}")

Original Count: 5 | Clean Count: 5


In [207]:
df_flat.filter(col("phone").isNull() | col("email").isNull()).show()

+-----------+-------------+---------+----------+----------+-------------+-----------------+------------+
|customer_id|customer_name|     city|membership|     phone|        email|preferred_service|budget_range|
+-----------+-------------+---------+----------+----------+-------------+-----------------+------------+
|          2|    Sana Khan|Bangalore|    Silver|      NULL|sana@mail.com|            Hotel|        NULL|
|          3|  John Mathew|     NULL|      Gold|9876500013|         NULL|           Flight|        High|
|          5|   Vikram Rao|   Mumbai|  Platinum|      NULL|         NULL|           Flight|        High|
+-----------+-------------+---------+----------+----------+-------------+-----------------+------------+



In [208]:
df_flat.filter(col("preferred_service").isNull() | col("budget_range").isNull()).show()

+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|customer_name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+
|          2|    Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          4| Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+-------------+---------+----------+----------+---------------+-----------------+------------+

